In [1]:
!pip install transformers torch scikit-learn -q

In [2]:
from google.colab import files

uploaded = files.upload()

Saving combined_en.csv to combined_en.csv


In [3]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

df = pd.read_csv('combined_en.csv')
print(df['label'].value_counts())

Device: cuda
label
rational_argument         2804
fear_appeal                701
emotional_manipulation     517
authority_appeal           470
demagogy_tricks            171
Name: count, dtype: int64


In [4]:
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

print(f'Classes: {le.classes_}')
print('\nMapping:')
for i, cls in enumerate(le.classes_):
    print(f'{i} -> {cls}')

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['label_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label_id'].values
)

print(f'\nTrain: {len(X_train)}')
print(f'Test: {len(X_test)}')

Classes: ['authority_appeal' 'demagogy_tricks' 'emotional_manipulation'
 'fear_appeal' 'rational_argument']

Mapping:
0 -> authority_appeal
1 -> demagogy_tricks
2 -> emotional_manipulation
3 -> fear_appeal
4 -> rational_argument

Train: 3730
Test: 933


In [5]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

class TextDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f'Train batch: {len(train_loader)}\nTest batch: {len(test_loader)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train batch: 234
Test batch: 59


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=5,
    ignore_mismatched_sizes=True
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Model Device: {next(model.parameters()).device}');

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params: 66,957,317
Trainable params: 66,957,317
Model Device: cuda:0


In [7]:
EPOCHS = 6
optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=50,
    num_training_steps=total_steps
)

print(f'Epochs: {EPOCHS}\nBatches/epoch: {len(train_loader)}\nTotal steps: {total_steps}')

Epochs: 6
Batches/epoch: 234
Total steps: 1404


In [8]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # clean grad
        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

def eval_epoch(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')

    return f1, all_preds, all_labels

In [9]:
import time
import copy

best_f1 = 0
best_model_state = None

torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    f1, preds, labels_true = eval_epoch(model, test_loader, device)
    elapsed = time.time() - start_time

    print(f'Epoch: {epoch+1}/{EPOCHS} | '
          f'Loss: {train_loss:.4f} | '
          f'Train Acc: {train_acc:.4f} | '
          f'F1: {f1:.4f} | '
          f'Time: {elapsed:.0f}s')

    if f1 > best_f1:
        best_f1 = f1
        best_model_state = copy.deepcopy(model.state_dict())
        print(f'F1={best_f1:.4f}')

print(f'\nBest F1: {best_f1:.4f}')

Epoch: 1/6 | Loss: 1.0739 | Train Acc: 0.6343 | F1: 0.4334 | Time: 53s
F1=0.4334
Epoch: 2/6 | Loss: 0.7000 | Train Acc: 0.7834 | F1: 0.5888 | Time: 47s
F1=0.5888
Epoch: 3/6 | Loss: 0.4469 | Train Acc: 0.8649 | F1: 0.6929 | Time: 49s
F1=0.6929
Epoch: 4/6 | Loss: 0.2966 | Train Acc: 0.9115 | F1: 0.7330 | Time: 49s
F1=0.7330
Epoch: 5/6 | Loss: 0.1925 | Train Acc: 0.9432 | F1: 0.7530 | Time: 50s
F1=0.7530
Epoch: 6/6 | Loss: 0.1369 | Train Acc: 0.9619 | F1: 0.7708 | Time: 49s
F1=0.7708

Best F1: 0.7708


In [11]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import os
os.makedirs('/content/drive/MyDrive/psycholinguistic_detector', exist_ok=True)

torch.save(best_model_state, '/content/drive/MyDrive/psycholinguistic_detector/distilbert_best.pt')
tokenizer.save_pretrained('/content/drive/MyDrive/psycholinguistic_detector/tokenizer')

import json
classes = le.classes_.tolist()
with open('/content/drive/MyDrive/psycholinguistic_detector/classes.json', 'w') as f:
  json.dump(classes, f)

print(f'Best F1: {best_f1:.4f}')
print(f'Classes: {classes}')

Mounted at /content/drive
Best F1: 0.7708
Classes: ['authority_appeal', 'demagogy_tricks', 'emotional_manipulation', 'fear_appeal', 'rational_argument']
